# Capítulo 3 — Metodologia de avaliação

**Livro:** começa na página **230** do PDF.

**Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Entropia · PDF p. 239

#### Exercício: Entropia de uma distribuição

Calcule a entropia (em bits) de duas distribuições: uma moeda honesta `[0.5, 0.5]` e uma moeda viciada `[0.9, 0.1]`. Qual tem mais "incerteza"? Use `H = -Σ p·log2(p)`.

In [ ]:
import numpy as np

def entropia(p):
    p = np.array(p, dtype=float)
    return float(-(p * np.log2(p)).sum())

print("moeda honesta :", entropia([0.5, 0.5]), "bits")
print("moeda viciada :", round(entropia([0.9, 0.1]), 3), "bits")

### Entropia Cruzada · PDF p. 241

#### Exercício: Entropia de uma distribuição

Calcule a entropia (em bits) de duas distribuições: uma moeda honesta `[0.5, 0.5]` e uma moeda viciada `[0.9, 0.1]`. Qual tem mais "incerteza"? Use `H = -Σ p·log2(p)`.

In [ ]:
import numpy as np

def entropia(p):
    p = np.array(p, dtype=float)
    return float(-(p * np.log2(p)).sum())

print("moeda honesta :", entropia([0.5, 0.5]), "bits")
print("moeda viciada :", round(entropia([0.9, 0.1]), 3), "bits")

### Perplexidade · PDF p. 242

#### Exercício: Perplexidade de uma resposta a partir dos logprobs

Peça uma resposta curta com `logprobs=True`. Calcule a **perplexidade** da resposta: `exp(-média dos logprobs)`. Compare uma frase previsível ("Complete: Batatinha quando nasce...") com uma pedida aleatória ("Escreva 8 palavras aleatórias").

In [ ]:
import math

def perplexidade(prompt):
    r = conversar([{"role": "user", "content": prompt}], max_tokens=20, logprobs=True, temperature=0)
    lps = [t.logprob for t in r.choices[0].logprobs.content]
    return r.choices[0].message.content, math.exp(-sum(lps) / len(lps))

for p in ["Complete o verso: Batatinha quando nasce", "Escreva 8 palavras aleatórias sem relação entre si."]:
    texto, ppl = perplexidade(p)
    print(f"{ppl:6.2f}  <- {texto!r}")

### Correção funcional · PDF p. 251

#### Exercício: O código gerado funciona? Teste com asserts

Peça ao modelo uma função Python `eh_primo(n)`. Execute o código gerado e teste com alguns casos (`assert`). Isso é **correção funcional** (p. 251).

Só rode código gerado por IA quando entender o que ele faz.

In [ ]:
codigo = perguntar(
    "Escreva só o código de uma função Python chamada eh_primo(n) que retorna True se n for primo. "
    "Sem explicações e sem blocos ```.", temperature=0)
print(codigo)
escopo = {}
exec(codigo, escopo)
eh_primo = escopo["eh_primo"]
casos = {2: True, 3: True, 4: False, 17: True, 1: False, 0: False, 25: False}
acertos = sum(eh_primo(n) == esperado for n, esperado in casos.items())
print(f"{acertos}/{len(casos)} casos corretos")

### Medidas de similaridade em relação aos dados de referência · PDF p. 253

#### Exercício: Comparar com uma resposta de referência

Para a pergunta *"O que é MRP?"*, escreva uma resposta de referência. Gere a resposta do modelo e compare com 3 medidas: **igualdade exata**, **similaridade lexical** (`difflib.SequenceMatcher`) e **similaridade semântica** (embeddings). Qual faz mais sentido?

In [ ]:
import difflib

referencia = "MRP é o planejamento das necessidades de materiais: calcula o que comprar ou produzir e quando."
resposta = perguntar("Em uma frase, o que é MRP na indústria?", temperature=0)
print("Modelo:", resposta)
print("Exata     :", resposta.strip() == referencia)
print("Lexical   :", round(difflib.SequenceMatcher(None, resposta.lower(), referencia.lower()).ratio(), 3))
v = embeddings([resposta, referencia])
print("Semântica :", round(float(similaridade(v[:1], v[1:])[0, 0]), 3))

### Introdução à incorporação · PDF p. 264

#### Exercício: Embeddings: frases parecidas ficam perto

Gere embeddings para 6 frases: 3 sobre estoque e 3 sobre futebol. Monte a matriz de similaridade e mostre com `plt.imshow`. As frases do mesmo assunto ficam mais próximas?

In [ ]:
import matplotlib.pyplot as plt

frases = ["O estoque de segurança acabou", "Precisamos repor o almoxarifado", "Há ruptura de materiais",
          "O time venceu a final", "O goleiro defendeu o pênalti", "O campeonato começa domingo"]
v = embeddings(frases)
m = similaridade(v, v)
plt.imshow(m, cmap="viridis"); plt.colorbar()
plt.xticks(range(6), range(1, 7)); plt.yticks(range(6), [f"{i+1}. {f[:25]}" for i, f in enumerate(frases)])
plt.title("Similaridade de cosseno"); plt.show()

### Como usar a IA como juiz · PDF p. 271

#### Exercício: Montar um juiz com nota de 1 a 5

Crie um prompt de **juiz** que recebe uma pergunta e uma resposta e devolve JSON com `nota` (1 a 5) e `justificativa`. Teste com uma resposta boa e uma ruim para a mesma pergunta.

In [ ]:
import json

JUIZ = ("Você avalia respostas. Critério: a resposta está correta e responde exatamente à pergunta? "
        "Devolva JSON com 'nota' (inteiro de 1 a 5) e 'justificativa' (uma frase).")

def julgar(pergunta, resposta):
    saida = perguntar(f"Pergunta: {pergunta}\nResposta: {resposta}", sistema=JUIZ, modelo=MODELO_FORTE,
                      temperature=0, response_format={"type": "json_object"})
    return json.loads(saida)

p = "Qual a diferença entre estoque de segurança e ponto de pedido?"
boa = ("Estoque de segurança é a reserva para cobrir variações; ponto de pedido é o nível de estoque "
       "que dispara uma nova compra, considerando a demanda no lead time mais o estoque de segurança.")
ruim = "São a mesma coisa, ambos indicam quanto vendemos por mês."
print("boa :", julgar(p, boa))
print("ruim:", julgar(p, ruim))

### Limitações da IA ​​como Juiz · PDF p. 276

#### Exercício: Viés de posição do juiz

Peça ao juiz para escolher a melhor entre duas respostas **A** e **B**. Depois troque a ordem. O juiz mantém a escolha? (p. 276–282 falam desse viés).

In [ ]:
def comparar(pergunta, r1, r2):
    return perguntar(f"Pergunta: {pergunta}\n\nResposta A: {r1}\n\nResposta B: {r2}\n\n"
                     "Qual resposta é melhor? Responda só A ou B.", modelo=MODELO_FORTE, temperature=0).strip()

p = "Por que usar temperatura baixa em um classificador?"
r1 = "Para as respostas ficarem mais estáveis e previsíveis."
r2 = "Porque com temperatura baixa a classificação tende a repetir a mesma resposta para a mesma entrada, o que facilita avaliar."
primeira = comparar(p, r1, r2)          # r1 em A
segunda = comparar(p, r2, r1)           # r1 em B
print("Ordem original -> vencedor:", "r1" if primeira == "A" else "r2")
print("Ordem trocada  -> vencedor:", "r1" if segunda == "B" else "r2")

## Classificação de modelos com avaliação comparativa · PDF p. 289

#### Exercício: Qual modelo ganha? Taxa de vitória

Escolha 4 perguntas. Responda cada uma com `MODELO` e com `MODELO_FORTE`. Use um juiz para escolher o melhor em cada par e calcule a **taxa de vitória** de cada modelo (p. 289–294).

In [ ]:
perguntas = ["O que é lead time?", "Explique overfitting em uma frase.",
             "Dê um exemplo de dado faltante no Titanic.", "Para que serve o RAG?"]
vitorias = {MODELO: 0, MODELO_FORTE: 0}
for p in perguntas:
    a, b = perguntar(p, modelo=MODELO, max_tokens=120), perguntar(p, modelo=MODELO_FORTE, max_tokens=120)
    voto = perguntar(f"Pergunta: {p}\n\nA: {a}\n\nB: {b}\n\nQual é melhor? Responda só A ou B.",
                     modelo=MODELO_FORTE, temperature=0).strip()
    vitorias[MODELO if voto == "A" else MODELO_FORTE] += 1
{m: f"{v}/{len(perguntas)}" for m, v in vitorias.items()}

## Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou